# genoselect — comprehensive demo

Run the cells top to bottom (**Shift+Enter**, or *Run All*). Requires `pip install genoselect` and a Python kernel where it is installed.

## 1. Dummy SNP dataset
Rows = individuals, columns = markers coded 0/1/2.

In [1]:
import tempfile
from pathlib import Path
import numpy as np
import genoselect as gs

rng = np.random.default_rng(1)
print('genoselect', gs.__version__)

n_ind, n_marker = 150, 400
freqs = rng.uniform(0.05, 0.95, size=n_marker)
geno = rng.binomial(2, freqs, size=(n_ind, n_marker)).astype(float)

qtl = rng.choice(n_marker, size=30, replace=False)
gv = (geno[:, qtl] - 2*freqs[qtl]) @ rng.normal(size=30)
gv = gv / gv.std()
pheno = gv + rng.normal(scale=np.sqrt((1-0.5)/0.5), size=n_ind)

geno[(rng.integers(0, n_ind, 60), rng.integers(0, n_marker, 60))] = np.nan
print(n_ind, 'individuals x', n_marker, 'markers;', int(np.isnan(geno).sum()), 'missing')

genoselect 0.2.0
150 individuals x 400 markers; 60 missing


## 2. Quality control & imputation

In [2]:
X, keep = gs.qc_markers(geno, maf=0.05, max_missing=0.10, impute=True)
print('after QC:', X.shape[1], 'markers; missing =', int(np.isnan(X).sum()))
geno_imp = gs.impute_markers(geno)
print('impute only, missing =', int(np.isnan(geno_imp).sum()))

after QC: 397 markers; missing = 0
impute only, missing = 0


## 3. Relationship matrix (VanRaden) + marker centring

In [3]:
G = gs.vanraden_grm(X)
print('GRM', G.shape, 'mean diag %.3f' % np.mean(np.diag(G)))
W, p, c, kept = gs.centre_markers(X)
print('centred markers', W.shape, 'c = %.1f' % c)

GRM (150, 150) mean diag 0.999
centred markers (150, 397) c = 146.4


## 4. GBLUP by REML (+ prediction)

In [4]:
test = rng.choice(n_ind, size=30, replace=False)
train = np.setdiff1d(np.arange(n_ind), test)

fit = gs.GBLUP().fit(X[train], pheno[train])
print('h2 = %.3f  (Vu=%.3f, Ve=%.3f)' % (fit.h2_, fit.Vu_, fit.Ve_))
pred = fit.predict(X[test])
print('test accuracy:', round(float(np.corrcoef(pred, pheno[test])[0,1]), 3))

# functional interface + raw REML solution
sol = gs.gblup(pheno[train], K=gs.vanraden_grm(X[train]))
print('reml h2:', round(sol['Vu']/(sol['Vu']+sol['Ve']), 3))
sol2 = gs.reml_solve(pheno[train], gs.vanraden_grm(X[train]))
print('lambda:', round(sol2['lam'], 3))

h2 = 0.639  (Vu=1.318, Ve=0.744)
test accuracy: 0.256
reml h2: 0.639


lambda:

 0.565


## 5. Model registry

In [5]:
print(gs.available_models())
enet = gs.make_model('elastic_net', random_state=1).fit(X[train], pheno[train])
print('elastic-net accuracy:', round(float(np.corrcoef(enet.predict(X[test]), pheno[test])[0,1]), 3))

['gblup', 'elastic_net', 'random_forest', 'gradient_boosting', 'ensemble']


elastic-net accuracy: 0.373


## 6. Cross-validation (k-fold, leave-group-out, benchmark)

In [6]:
cv = gs.cross_validate(X, pheno,
        models=['gblup','elastic_net','random_forest','gradient_boosting'],
        k=5, random_state=1)
cv.summary()

,model,mean,sd,n_folds
0,elastic_net,0.473006,0.166939,5
1,gblup,0.279505,0.122137,5
2,random_forest,0.273006,0.197730,5
3,gradient_boosting,0.222502,0.165894,5


In [7]:
groups = np.tile(np.arange(5), n_ind//5 + 1)[:n_ind]
gs.cross_validate(X, pheno, models=['gblup','elastic_net'],
                  scheme='leave_group_out', groups=groups).summary()

,model,mean,sd,n_folds
0,elastic_net,0.487892,0.194350,5
1,gblup,0.309902,0.113369,5


In [8]:
gs.benchmark(X, pheno, k=5, random_state=1).summary()

,model,mean,sd,n_folds
0,elastic_net,0.473006,0.166939,5
1,gblup,0.279505,0.122137,5
2,random_forest,0.273006,0.197730,5
3,gradient_boosting,0.222502,0.165894,5


## 7. Stacked super-learner ensemble

In [9]:
ens = gs.StackedEnsemble(random_state=1).fit(X[train], pheno[train])
print('weights:', {k: round(v,3) for k,v in ens.weights_.items()})
print('ensemble accuracy:', round(float(np.corrcoef(ens.predict(X[test]), pheno[test])[0,1]), 3))

weights: {'gblup': np.float64(0.107), 'elastic_net': np.float64(0.847), 'random_forest': np.float64(0.039), 'gradient_boosting': np.float64(0.007)}
ensemble accuracy: 0.367


## 8. Built-in simulator

In [10]:
pop = gs.simulate_population(n=120, m=500, n_qtl=50, h2=0.5, seed=1)
print(pop.geno.shape, 'GBLUP h2 =', round(gs.GBLUP().fit(pop.geno, pop.pheno).h2_, 3))

(120, 500) GBLUP h2 = 0.546


## 9. Reading real files (dummy VCF; PLINK/HapMap identical)

In [11]:
lines = [
    "##fileformat=VCFv4.2",
    "#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tA\tB\tC",
    "1\t10\trs1\tA\tG\t.\t.\t.\tGT\t0/0\t0/1\t1/1",
    "1\t20\trs2\tC\tT\t.\t.\t.\tGT\t1/1\t0/0\t0/1",
]
with tempfile.TemporaryDirectory() as d:
    pth = Path(d) / "dummy.vcf"
    pth.write_text("\n".join(lines) + "\n")
    gd = gs.read_vcf(pth)
print(gd, gd.samples, gd.markers)
print(gd.geno)
# gs.read_plink("prefix"); gs.read_hapmap("x.hmp.txt")

<GenotypeData: 3 samples x 2 markers> ['A', 'B', 'C'] ['rs1', 'rs2']
[[0. 2.]
 [1. 0.]
 [2. 1.]]


**Done.** Every public genoselect function is exercised above.